# Deliverable 2 — EDA, Data Leakage & Choosing the Right Model

**ADSC × American Airlines — Fall 2026**

In Deliverable 1 you were handed one model — `XGBRegressor` — and you trained it. It worked, and you got an R² somewhere north of 0.97.

Two questions were never asked:

1. **Was that score real?** (You cannot currently answer that. Section 3.)
2. **Was XGBoost the right choice?** (You have no idea, because you never tried anything else.)

This deliverable answers both. You will go deeper on EDA, learn what **data leakage** is and measure what it cost you, then run a genuine model-selection exercise: **Lasso vs Ridge**, **Random Forest vs XGBoost**, and **ARIMA** — plus the choice that sits underneath all of them and that nobody teaches you to make deliberately, your **loss function**.

---

## What you will learn

| Section | Topic | The question it answers |
|---|---|---|
| **2** | **Deeper EDA** | What does the data actually look like, and what does that imply about which model to reach for? |
| **3** | **Data leakage** | Why was the Deliverable 1 score too good, and how do I stop it happening again? |
| **4** | **Lasso vs Ridge** | Two linear models, two penalties. When do I want shrinkage, and when do I want feature selection? |
| **5** | **Loss functions** | MSE, MAE, Huber, quantile — what am I actually asking the model to minimise, and does it match what the business cares about? |
| **6** | **XGBoost vs Random Forest vs ARIMA** | Three model families. Which fits *this* problem, and why is one of them answering a completely different question? |
| **7** | **Pick one and defend it** | Given everything above, what would you ship to American Airlines? |

The single most important idea in here: **model selection is not a leaderboard.** It is a set of choices — model family, loss function, regularisation — each of which should follow from something you observed in the data or something the business told you. By the end you should be able to justify every one of your choices without using the phrase "it scored the best".

---

## What we are building on from Deliverable 1

We assume all of this and will not re-explain it:

- loading the CSV, `.shape` / `.info()` / `.describe()`
- null counts, mean / median / std / quartiles / IQR with NumPy
- the 1.5 × IQR outlier rule
- the price histogram, price-by-airline box plot, duration-vs-price scatter
- `LabelEncoder`, an 80/20 `train_test_split`, `XGBRegressor`
- MAE, RMSE, R², and gain-based feature importance

Keep **your own completed Deliverable 1 notebook** open in another tab — it is in `deliverable1/`, saved under your name — because we refer back to it by section number throughout.

---

## The git workflow changed — read this before you touch anything

Deliverable 1 was solo: you cut `deliverable1/<your-name>` and opened a PR straight into `main`. With 70+ members that does not scale, and it is not how real teams work either.

**From Deliverable 2 onward you work in a team, and your team owns a branch tree.**

```
main                          <- protected. Only a PM merges here.
 └── team-06/main             <- YOUR TEAM's integration branch ("team trunk")
      ├── team-06/alex-chen    <- a member branch: your own work
      ├── team-06/priya-r      <- another member, in parallel
      └── team-06/sam-lee
```

**What matters is the structure, not the names.** Three rules:

1. **Your team has one trunk branch.** The suggested name is `team-##/main` — your team number, zero-padded, so team 6 is `team-06`. If that exact name is unavailable (see the note below), any name your team agrees on is completely fine.
2. **You personally work on your own branch off that trunk**, e.g. `team-##/<your-name>`. Never commit directly to the trunk.
3. **Work moves up the tree:** your branch → your team's trunk → `main`. Your PR goes into **the trunk**, reviewed by a teammate. Only the trunk opens a PR into `main`, and only a PM merges that one.

### Setting it up

**One person on the team** creates the trunk (do this once, then tell your teammates it exists):

```sh
git switch main
git pull origin main
git switch -c team-06/main          # use YOUR team number
git push -u origin team-06/main
```

**Every member** then branches off the team trunk — not off `main`:

```sh
git fetch origin
git switch -c team-06/alex-chen origin/team-06/main
git push -u origin team-06/alex-chen
```

### Submitting

```sh
git add deliverable2/flight_price_modeling.ipynb
git commit -m "deliverable2: EDA, leakage fixes, model selection"
git push

# then on GitHub: open a PR from team-06/alex-chen  ->  team-06/main
# (base branch is your TEAM's trunk, NOT the repo's main)
```

> **⚠️ If `git switch -c team-##/main` fails, this is why.** Git stores branch names as file paths, so a branch name cannot be a *prefix* of another. If a plain branch named `team-07` already exists, then `team-07/main` is **impossible** — git says `cannot lock ref ... 'team-07' exists`. Several teams are in exactly this position from earlier in the term.
>
> **This is not a problem, and nothing needs deleting.** Just pick a name that is free and tell your team what it is:
>
> ```sh
> git switch -c team-07-main      # a dash instead of a slash -- always works
> ```
>
> The same rule applies to member branches: `team-06/alex-chen` and `team-06/alex-chen-v2` are fine, but `team-06/alex-chen/fix` would be blocked by `team-06/alex-chen`. Keep each name a flat leaf.

> **⚠️ Notebooks and merge conflicts.** A `.ipynb` is JSON, and git conflicts on it are genuinely horrible. Two habits make it survivable:
> - **Clear outputs before every commit** (Kernel → Restart & Clear Output). Most notebook conflicts are rendered output, not code.
> - **Each member completes their own copy of this notebook.** You are *not* meant to split the cells across teammates. The notebook is individual learning; the shared team artifact is `deliverable2/TEAM_REPORT.md`, which your team fills in together on the team trunk.

Full detail, including each member's named review duty, is in [`deliverable2/README.md`](README.md). The general workflow rules have not changed — [`CONTRIBUTING.md`](../CONTRIBUTING.md) is still the operating manual.

---

> ### 🗒️ A note on AI tools
>
> Half of this deliverable is *judgment*: is this leakage, is Lasso the right penalty, does ARIMA even apply here. An AI will hand you a confident answer to every one of those, and you will have learned nothing and be unable to defend it when a teammate asks in review.
>
> **Use AI as a tutor, not an author:**
> - "Why does Lasso drive coefficients to exactly zero when Ridge does not?"
> - "What does `objective='reg:pseudohubererror'` change about training?"
> - "Why is ARIMA inappropriate for data without a time index?"
>
> **You must be able to explain every line you submit.** A teammate *will* review your PR and *will* ask.

---

## How to approach this deliverable

Read this before you start. It will save you more time than it costs.

### 1. The written answers are the deliverable — not the plots

Every section ends with a markdown prompt asking *why*. Those are the thing being assessed. A plot with no conclusion attached is worth nothing, and a number with no interpretation is worth less than no number at all, because it invites someone to act on it.

Concretely: when a cell tells you to make a histogram, the deliverable is not the histogram. It is the sentence **"...therefore I should \_\_\_ when I model."** If you cannot finish that sentence, you have not finished the cell.

### 2. The honest score is the goal, whichever direction it moves

Section 3 removes three leaks from your Deliverable 1 setup and measures what each was worth. **Do not assume the number must fall.** On some datasets a leak is worth a lot of fake R² and the honest score drops sharply; on others it turns out to have been adding almost nothing, and your score barely moves or even ticks up.

Both outcomes are real results. **The point is not that your score goes down — it is that you can no longer tell which it will be until you remove the leaks and measure.** An unmeasured score is untrustworthy no matter how good it looks.

So: report whatever you get, and never tune your way toward a number you were hoping for. If your honest R² does drop, being able to explain why the lower number is the real one is the single most professional thing a data scientist does. If it does not drop, say so and explain why that leak mattered less here than it might have.

### 3. How to think about choosing a model

The instinct from Deliverable 1 is "run them all, keep the best score". Resist it. Work through the questions in this order, because each one can eliminate options before you spend compute on them:

1. **What shape is my data, and what shape does this model require?** This is first because it is disqualifying. ARIMA needs an ordered time series; if you do not have one, no amount of tuning helps. Section 6 is built around this question.
2. **What am I asking the model to minimise?** The default is squared error, which is a choice you have been making silently. Section 5 is about making it on purpose.
3. **What does the stakeholder need?** A client who needs to *explain* a price to a customer may be better served by a Ridge model with readable coefficients than by a boosted ensemble that scores two points higher. Accuracy is one requirement among several.
4. **What does it cost?** Fit time, tuning burden, how many ways it can break in production. A model that takes 40× longer for 1% more R² is usually the wrong call.
5. **Only then: what scores best?**

If your final justification is "it had the highest R²", you have skipped the deliverable.

### 4. How to work the code

- **Run cells in order, top to bottom.** Later sections depend on variables defined earlier (`clean`, `X_train`, `prep_tree`). If something is `None` or undefined, you skipped a cell.
- **Check your work as you go.** After any transform, look at the result — `.head()`, `.shape`, `.value_counts()`. Most bugs in data code are silent: the wrong answer computes perfectly happily.
- **Some cells are slow.** Random Forest on 240k rows and 5-fold CV both take minutes. If a cell is painful on your machine, **subsample it and say so in your PR** — that is a normal engineering decision, not cheating. `X.sample(80_000, random_state=RANDOM_STATE)` is your friend.
- **Read the error, then the line above it.** Python tracebacks name the failing line. Nine times in ten the actual mistake is one line earlier.
- **Commit per section**, not in one lump at the end. Small commits are the entire reason your history is useful to a reviewer.

### 5. How to work with your team

Each member completes **their own full copy** of this notebook — the work is individual, because the learning is individual. What your team does **together** is agree on the conclusions and write them into `TEAM_REPORT.md` on your team's trunk branch.

Beyond that, how your team organises itself is up to your team. Decide in your first meeting and write the decision down somewhere your PM can see it.

One piece of hard-won advice: **clear your outputs before every commit** (Kernel → Restart & Clear Output). Notebook merge conflicts are mostly conflicts in rendered output, not in code, and this one habit prevents the large majority of them.

---

## References

Every link below was checked and goes to the specific page you need — not a search result or a landing page. Keep the ones for your current section open in browser tabs.

### Start here

| | |
|---|---|
| **Dataset** — Flight Price Prediction (EaseMyTrip) | https://www.kaggle.com/datasets/shubhambathwal/flight-price-prediction |
| **scikit-learn: Common pitfalls and recommended practices** — the single most useful page for this deliverable. Shows a "Wrong" and a "Right" version of each mistake. | https://scikit-learn.org/stable/common_pitfalls.html |
| **An Introduction to Statistical Learning** — the standard undergraduate text, **free PDF**. Ch. 6 covers Ridge/Lasso, Ch. 8 covers trees and boosting. Python edition (ISLP) also available. | https://www.statlearning.com/ |

### Section 3 — leakage & validation

| | |
|---|---|
| Data leakage, with a worked example of a score collapsing from 0.76 to chance once the leak is fixed | https://scikit-learn.org/stable/common_pitfalls.html#data-leakage |
| Inconsistent preprocessing — why the transform must be fitted once and applied, not refitted | https://scikit-learn.org/stable/common_pitfalls.html#inconsistent-preprocessing |
| Pipelines and composite estimators (`Pipeline`, `ColumnTransformer`) | https://scikit-learn.org/stable/modules/compose.html |
| Cross-validation: evaluating estimator performance | https://scikit-learn.org/stable/modules/cross_validation.html |

### Section 4 — Lasso vs Ridge

| | |
|---|---|
| Ridge regression — §1.1.2 of the Linear Models guide | https://scikit-learn.org/stable/modules/linear_model.html#ridge-regression-and-classification |
| Lasso — §1.1.3, including the note on why the objective is scaled by `1/(2·n_samples)` | https://scikit-learn.org/stable/modules/linear_model.html#lasso |
| Elastic-Net — §1.1.5 | https://scikit-learn.org/stable/modules/linear_model.html#elastic-net |
| **Common pitfalls in interpreting linear coefficients** — worked example of why scale matters, why coefficients are conditional not marginal, and how correlated features make coefficients unstable. Directly relevant to reading your Lasso output. | https://scikit-learn.org/stable/auto_examples/inspection/plot_linear_model_coefficient_interpretation.html |

### Section 5 — loss functions

| | |
|---|---|
| XGBoost learning-task parameters — the full `objective` list (`reg:squarederror`, `reg:absoluteerror`, `reg:pseudohubererror`, `reg:quantileerror`) | https://xgboost.readthedocs.io/en/stable/parameter.html#learning-task-parameters |
| **Prediction intervals for gradient boosting regression** — the reference implementation for Section 5's quantile models, including the coverage check. Note its own conclusion: *each model performs best on the metric it was trained for.* | https://scikit-learn.org/stable/auto_examples/ensemble/plot_gradient_boosting_quantile.html |
| Regression metrics — MAE, RMSE, R², and `mean_pinball_loss` | https://scikit-learn.org/stable/modules/model_evaluation.html#regression-metrics |

### Section 6 — model families

| | |
|---|---|
| **Introduction to Boosted Trees** — the clearest explanation of additive training anywhere, and it explicitly notes that random forests and boosted trees share the same model form and differ in *how they are trained*. Read this before writing your bagging-vs-boosting answer. | https://xgboost.readthedocs.io/en/stable/tutorials/model.html |
| Random forests and randomized tree ensembles — §1.11.2 | https://scikit-learn.org/stable/modules/ensemble.html#random-forests |
| Gradient-boosted trees — §1.11.1 | https://scikit-learn.org/stable/modules/ensemble.html#gradient-boosted-trees |
| `statsmodels` ARIMA API — the `order=(p, d, q)` signature | https://www.statsmodels.org/stable/generated/statsmodels.tsa.arima.model.ARIMA.html |
| **Forecasting: Principles and Practice, Ch. 9 — ARIMA models.** Free online, and the standard reference. | https://otexts.com/fpp3/arima.html |
| §9.1 Stationarity and differencing — read this before choosing your `d` | https://otexts.com/fpp3/arima.html#sec-stationarity |

---

## Setup

Copy the dataset you already downloaded to **`deliverable2/data/Clean_Dataset.csv`**. That path is git-ignored; **do not commit the CSV.**

One new package this time — `statsmodels`, for ARIMA:

```sh
pip install pandas numpy matplotlib seaborn scikit-learn xgboost statsmodels
```

---
# Section 0 — Imports & helpers

Run this cell first. Everything below depends on it.

Some cells in this notebook are **given to you complete** — the helpers, and the reproduction of your Deliverable 1 baseline. Read them; they are the reference implementations you will copy patterns from. The cells marked `# TODO` are yours.

In [ ]:
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# --- splitting & validation ------------------------------------------------
from sklearn.model_selection import train_test_split, KFold, cross_validate

# --- preprocessing ---------------------------------------------------------
# OneHotEncoder  : one 0/1 column per category. No invented ordering.
# OrdinalEncoder : integers in an order YOU specify (zero < one < two_or_more).
# StandardScaler : centres and scales numerics. Optional for trees,
#                  MANDATORY for Lasso/Ridge -- Section 4 explains why.
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

# --- composition -----------------------------------------------------------
# Pipeline guarantees every transform is fitted on the TRAINING fold only.
# That is our main defence against the leakage in Section 3.
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# --- the models we will compare -------------------------------------------
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge, Lasso, ElasticNet, LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor

# --- metrics ---------------------------------------------------------------
from sklearn.metrics import (mean_absolute_error, mean_squared_error,
                             r2_score, mean_pinball_loss)

# --- display ---------------------------------------------------------------
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 5)
sns.set_theme(style='whitegrid')

RANDOM_STATE = 42   # every split, every model. Reproducibility is not optional.
DATA_PATH = 'deliverable2/data/Clean_Dataset.csv'   # relative to the repo root

print('pandas', pd.__version__, '| numpy', np.__version__)
import sklearn; print('scikit-learn', sklearn.__version__)
import xgboost; print('xgboost', xgboost.__version__)

# statsmodels is only needed in Section 6c (ARIMA). If this import fails,
# `pip install statsmodels` and restart the kernel.
import statsmodels.api as sm
print('statsmodels', sm.__version__)

In [ ]:
def regression_report(y_true, y_pred, label=''):
    """Return a dict of the four metrics we use everywhere in this notebook.

    MAE   -- average absolute miss, in rupees. Interpretable.
    RMSE  -- squared-error based, so large misses dominate. RMSE >> MAE means a
             few very bad predictions are hiding behind a decent average.
    R2    -- fraction of price variance explained.
    MAPE% -- average miss as a PERCENTAGE of true price. The business metric: being
             off by 2,000 INR is trivial on a 52,000 ticket and awful on a 6,000 one.
    """
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return {
        'model': label,
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'R2': r2_score(y_true, y_pred),
        'MAPE%': float(np.mean(np.abs((y_true - y_pred) / y_true)) * 100),
    }


# Running scoreboard. Every model we fit goes on here, so at the end we can read
# the whole deliverable as one comparison table.
SCOREBOARD = []

def record(result):
    """Append a regression_report dict to the scoreboard and show the table."""
    SCOREBOARD.append(result)
    return pd.DataFrame(SCOREBOARD).set_index('model').round(4)


def show_scoreboard():
    return pd.DataFrame(SCOREBOARD).set_index('model').round(4)

print('Helpers defined: regression_report, record, show_scoreboard')

---
# Section 1 — Load the data

Nothing new here conceptually — but note the two given lines after the load, and read the comment attached to them.

In [ ]:
raw = pd.read_csv(DATA_PATH)
print('shape:', raw.shape)
print(raw.columns.tolist())

In [ ]:
# Two housekeeping fixes, given to you.
#
# 1. The CSV as published ships a nameless first column that pandas labels
#    'Unnamed: 0'. It is the row number from whoever exported the file -- somebody
#    ran to_csv() without index=False. It is NOT a feature: a flight you are asked
#    to price next week has no row number in your training file. Drop it.
#    (Deliverable 1 never dropped it, which means it was sitting in your X. Section 3
#    measures what that cost you.)
#
# 2. The Deliverable 1 schema table used 'Source City' / 'Price'; the actual CSV
#    uses 'source_city' / 'price'. Documentation drifting from the file is routine.
#    We lowercase everything so the rest of the notebook works either way.

df = raw.copy()
df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]
df = df.drop(columns=[c for c in df.columns if c.startswith('unnamed')])

print('shape:', df.shape)
print('columns:', df.columns.tolist())
df.head(3)

---
# Section 2 — EDA that informs a modelling decision

**Goal:** go deeper than Deliverable 1, and tie every observation to a modelling consequence.

Deliverable 1's EDA was descriptive: here is the distribution, here are the categories, here is a scatter plot. Useful, but it stopped short. **EDA earns its keep when it changes what you do next.** So for every plot in this section, the deliverable is not the plot — it is the sentence "...therefore I should \_\_\_ when I model".

Four things we need to know before choosing a model:

| # | What we look at | The modelling decision it drives |
|---|---|---|
| 2a | How `price` is distributed | Which **loss function** (Section 5) and whether to transform the target |
| 2b | Whether relationships are **linear** | Whether a linear model (Section 4) can compete at all |
| 2c | The **booking curve** over `days_left` | Whether a **time-series** model (Section 6) has anything to work with |
| 2d | Category counts and dominance | Which **encoding**, and whether one feature will swamp everything |

---

### 2a — The shape of the target

Deliverable 1 plotted the price histogram and noted it was right-skewed. Go further: plot it **split by class**.

In [ ]:
# 1. price by class
sns.histplot(data=df, x='price', hue='class', bins=80, element='step')
plt.title('Price by class'); plt.xlabel('price (INR)')
plt.show()

# 2. the same on a log x-axis
sns.histplot(data=df, x='price', hue='class', bins=80, element='step')
plt.xscale('log')
plt.title('Price by class (log x-axis)'); plt.xlabel('price (INR, log scale)')
plt.show()

# 3. skewness, raw vs log
print('skew raw      :', round(df['price'].skew(), 3))
print('skew log1p    :', round(np.log1p(df['price']).skew(), 3))

# 4. mean / median per class
df.groupby('class')['price'].agg(['count', 'mean', 'median', 'std']).round(0)

**What you should see, and what it means.** *(3–4 sentences.)*

The distribution is not one skewed hump — it is **two separate populations**, Economy and Business, with Business roughly an order of magnitude more expensive.

That matters for Section 5 in a concrete way. Squared error (MSE) penalises a ₹5,000 miss identically whether the ticket costs ₹6,000 or ₹52,000 — but the first is an 83% error and the second is under 10%. So **a model trained on MSE will quietly prioritise getting Business right**, because that is where the big absolute numbers are.

Write down: the two class means, the raw skew, the log skew, and one sentence on which loss function you now expect to prefer and why.

Business averages **₹52,540** (median ₹53,164) and Economy **₹6,572** (median ₹5,772), about 8× apart. The raw price skew is **1.06**, and after `log1p` it falls to **0.41**, so the log scale is much more symmetric, which hints that modelling log(price) would be reasonable. Because squared error is dominated by the big Business numbers, I now expect a loss that respects relative error (MAE, or a log target) to suit a business goal better than plain MSE, and I will report MAPE alongside rupee errors.

### 2b — Is this problem linear?

Before you can argue that you need a boosted tree ensemble, you have to know what a straight line gets you. And before you can read a correlation matrix, note the limit of the tool: **Pearson correlation measures only the *linear* part of a relationship.** A perfect U-shaped relationship has a correlation near zero. A low correlation does not mean "no relationship" — it means "no *linear* relationship", which is a much narrower claim.

In [ ]:
num = df[['duration', 'days_left', 'price']]
tmp = num.copy()
tmp['is_business'] = (df['class'] == 'Business').astype(int)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.heatmap(num.corr(), annot=True, cmap='coolwarm', center=0, ax=axes[0])
axes[0].set_title('numeric columns only')
sns.heatmap(tmp.corr(), annot=True, cmap='coolwarm', center=0, ax=axes[1])
axes[1].set_title('plus is_business')
plt.tight_layout(); plt.show()

s = df.sample(5000, random_state=RANDOM_STATE)
sns.regplot(data=s, x='duration', y='price',
            scatter_kws={'s': 4, 'alpha': 0.3}, line_kws={'color': 'red'})
plt.title('price vs duration with a straight-line fit')
plt.show()

**Your read.** *(3–4 sentences. Which feature carries the most linear signal? Where does the straight line visibly fail? Does the `duration` correlation being weak prove duration does not matter — and if not, why not?)*

`is_business` has a correlation of **0.94** with price, far above `duration` (0.20) and `days_left` (−0.09), so class carries nearly all of the linear signal. The straight line through the duration-vs-price scatter visibly fails: the cloud is two bands (Economy low, Business high) and one line runs between them and fits neither. A weak `duration` correlation does not prove duration is unimportant: Pearson only measures *linear* association, and duration's effect is tangled up with class and stops, so within a single class it may matter more than it does overall.

### 2c — The booking curve: the one piece of this dataset with a time axis

This is the most important plot in the notebook, and it is the one Deliverable 1 never made.

`days_left` is **days until departure**. It is not a timestamp, but it *is* an ordered horizon — so if you average price at each value of `days_left`, you get a genuine **sequence**: how fare moves as departure approaches. Airlines call this the booking curve, and forecasting it is a real job.

That ordered sequence is the only thing in this dataset a time-series model like ARIMA can operate on. Section 6c comes back to it.

In [ ]:
curve = df.groupby('days_left')['price'].agg(['mean', 'median', 'count'])

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].plot(curve.index, curve['mean'], marker='o', ms=3)
axes[0].invert_xaxis()
axes[0].set_xlabel('days until departure  (departure is at the right)')
axes[0].set_ylabel('mean price (INR)'); axes[0].set_title('booking curve, all flights')

for cabin, grp in df.groupby('class'):
    c = grp.groupby('days_left')['price'].mean()
    axes[1].plot(c.index, c.values, marker='o', ms=3, label=cabin)
axes[1].invert_xaxis(); axes[1].legend()
axes[1].set_xlabel('days until departure'); axes[1].set_title('booking curve by class')
plt.tight_layout(); plt.show()

print('points in the aggregate curve:', len(curve))
curve.head(3), curve.tail(3)

**Read the curve.** *(4–5 sentences.)*

- Is the relationship between `days_left` and price **linear**? At roughly what point does it turn?
- Economy vs Business: same shape, or different?
- This answers something important about Section 4. A linear model given raw `days_left` fits **one straight line** through this curve. Looking at the shape, how badly will that go — and what could you do to a linear model to let it cope? *(Think: what if `days_left` were bucketed, or log-transformed, or squared?)*
- How many data points does the aggregate curve have? Is that a lot or a little for fitting a time-series model?

- **Not linear.** For Economy the fare is flat at about ₹4,750–5,300 from 49 days out down to roughly 20 days, then climbs: ₹11,188 at 10 days, ₹13,174 at 3 days and ₹14,613 at 1 day. The turn is roughly 15–20 days before departure.
- **Economy vs Business:** same shape (flat, then steep near departure) but very different scale. Economy roughly triples in the last weeks; Business goes from about ₹51,000 to about ₹65,000.
- **For a linear model:** a single straight line through a flat-then-steep curve will over-predict far out and under-predict near departure. I could bucket `days_left` (e.g. 1–3, 4–7, 8–14, 15–30, 30+), add a squared or log term, or use a spline.
- **Sample size:** the aggregate curve has only **49 points**, which is very little for fitting a time-series model.

### 2d — Categories: counts, dominance, and encoding

In [ ]:
cats = ['airline', 'source_city', 'departure_time', 'stops',
        'arrival_time', 'destination_city', 'class']
for c in cats:
    print(f'{c:20s} {df[c].nunique():3d}  {sorted(df[c].unique())}')

print()
print(pd.crosstab(df['airline'], df['class'], values=df['price'], aggfunc='mean').round(0))

route_price = df.groupby(['source_city', 'destination_city'])['price'].mean()
print()
print('cheapest routes:'); print(route_price.sort_values().head(5).round(0))
print('priciest routes:'); print(route_price.sort_values().tail(5).round(0))

> **Which columns are ordered, and which are not?** This matters in Section 4 more than you would guess.
>
> - `stops` is **ordinal**: `zero < one < two_or_more`. That is a real ranking.
> - `departure_time` / `arrival_time` are **ordinal**: `Early_Morning < Morning < Afternoon < Evening < Night < Late_Night` — they are a clock.
> - `airline`, `source_city`, `destination_city` are **nominal**. There is no sense in which Vistara > SpiceJet.
>
> Deliverable 1 used `LabelEncoder` on all seven, which assigns integers **alphabetically**. For `stops` that gives `one=0, two_or_more=1, zero=2` — telling the model a direct flight ranks *above* a two-stop flight. Not just arbitrary: backwards.
>
> A tree can route around this by spending extra splits to separate the categories again, so the cost shows up as wasted capacity. **A linear model cannot route around it at all** — it gets one coefficient per column and is forced to assume evenly-spaced, ordered effects. Section 4 shows you exactly how much that costs.

### 2e — EDA summary: four decisions

Fill this in before you model. This is the deliverable for Section 2 — not the plots.

| Observation | Therefore, when I model, I will... |
|---|---|
| Target distribution is **two populations, Economy (mean ₹6,572) and Business (mean ₹52,540), right-skewed (skew 1.06, log skew 0.41)** | judge models by MAE in rupees **and** MAPE (relative error), not only squared-error metrics, and consider a log target or MAE-type loss |
| The strongest single predictor appears to be **class (correlation 0.94 with price, vs 0.20 for duration)** | always keep `class` as a feature, check it survives every preprocessing step (the `remainder='drop'` footgun), and look at errors per class |
| The `days_left` relationship is **non-linear: flat until about 15–20 days out, then a steep rise** | not trust a linear model on raw `days_left`; use trees, or bucket/transform it for linear models |
| Ordered categories are **`stops` (zero < one < two_or_more) and the time bands (a clock)**; airline and cities are nominal | encode `stops` and the time bands with an `OrdinalEncoder` in an explicit order, and one-hot the nominal columns, instead of alphabetical `LabelEncoder` |

---
# Section 3 — Data leakage

**Goal:** find the three places the Deliverable 1 setup let information leak, fix each one, and **measure what each was actually worth** rather than assuming.

> **Leakage** is when information reaches the model at training time that will not be available at prediction time.

It is not exotic. It is the default result of writing code in the order that feels natural — "clean everything, then split" — because every transform fitted before the split has already seen the test set.

Deliverable 1 had three leaks:

| # | Leak | Where |
|---|---|---|
| 1 | **The row counter.** `Unnamed: 0` was never dropped, so it sat in `X`. A future flight has no row number. | D1 §3a |
| 2 | **Duplicate rows across the split.** Once the unique counter is removed, this dataset contains repeated rows; copies can land in train *and* test, so the model is graded on rows it memorised. | D1 §3c |
| 3 | **Encoders fitted on the full dataset.** `df[col] = le.fit_transform(df[col])` ran in §3b, *before* the split in §3c. | D1 §3b |

**Be precise about leak 3's severity**, because overstating it is its own error. A plain `LabelEncoder` on complete categorical columns learns a list of category *names* — it learns nothing about price, so the practical damage is small. It is still **wrong as a habit**, because the identical code shape is severe the moment the transform learns something numeric:

- a `StandardScaler` fitted pre-split leaks the test set's mean and variance
- a **target encoder** fitted pre-split leaks the test set's *prices* directly
- an imputer fitted pre-split leaks the test distribution

You will not remember which transforms are safe at 2am. **Fit every transform inside the split, always.** That is what `Pipeline` is for.

### 3a — Reproduce the Deliverable 1 baseline (given)

Given to you complete, so every member starts from the identical reference number. Read it and match each line to a section of the old notebook.

In [ ]:
def deliverable1_baseline(path, random_state=RANDOM_STATE):
    """Reproduce Deliverable 1 exactly -- leaks and all.

      D1 S1  : read_csv
      D1 S2a : dropna
      D1 S3a : drop 'flight'        <- but NOT the row counter
      D1 S3b : LabelEncoder on the FULL frame, before the split   <- leak 3
      D1 S3c : X = everything except price; 80/20 split           <- dupes straddle it
      D1 S4  : XGBRegressor, same hyperparameters
    """
    from sklearn.preprocessing import LabelEncoder

    frame = pd.read_csv(path)
    frame.columns = [c.strip().lower().replace(' ', '_') for c in frame.columns]
    frame = frame.dropna().drop(columns=['flight'])

    for col in ['airline', 'source_city', 'departure_time', 'stops',
                'arrival_time', 'destination_city', 'class']:
        # Fitted on 100% of rows -- including the ones we will call "unseen".
        frame[col] = LabelEncoder().fit_transform(frame[col])

    X = frame.drop(columns=['price'])
    y = frame['price']
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, random_state=random_state)

    model = XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.1,
                         subsample=0.8, colsample_bytree=0.8,
                         random_state=random_state, n_jobs=-1)
    model.fit(X_tr, y_tr)
    return model, X_tr, X_te, y_tr, y_te, model.predict(X_te)


t0 = time.time()
d1_model, d1_X_tr, d1_X_te, d1_y_tr, d1_y_te, d1_pred = deliverable1_baseline(DATA_PATH)
print(f'trained in {time.time() - t0:.1f}s')
print('features the D1 model was given:', d1_X_tr.columns.tolist())

record(regression_report(d1_y_te, d1_pred, 'D1 baseline (3 leaks)'))

In [ ]:
imp = pd.Series(d1_model.feature_importances_, index=d1_X_tr.columns).sort_values(ascending=False)
print(imp.round(4))

raw_sorted = pd.read_csv(DATA_PATH)
raw_sorted.columns = [c.strip().lower().replace(' ', '_') for c in raw_sorted.columns]
raw_sorted.groupby('class')['unnamed:_0'].agg(['min', 'max', 'count'])

### 3b — Remove the leaks, one at a time

One change per step, so each result is **attributable**. Change three things at once and you learn nothing about any of them.

In [ ]:
def d1_variant(path, drop_counter=True, dedupe=False, random_state=RANDOM_STATE):
    """The Deliverable 1 recipe with the leaks switched off one at a time.
    Everything else (LabelEncoder on the full frame, hyperparameters, split seed) is
    identical to deliverable1_baseline, so each change is attributable."""
    from sklearn.preprocessing import LabelEncoder

    frame = pd.read_csv(path)
    frame.columns = [c.strip().lower().replace(' ', '_') for c in frame.columns]
    frame = frame.dropna().drop(columns=['flight'])
    if drop_counter:
        frame = frame.drop(columns=['unnamed:_0'])
    if dedupe:
        before = len(frame)
        frame = frame.drop_duplicates()
        print('dropped', before - len(frame), 'duplicate rows')

    for col in ['airline', 'source_city', 'departure_time', 'stops',
                'arrival_time', 'destination_city', 'class']:
        frame[col] = LabelEncoder().fit_transform(frame[col])

    X_ = frame.drop(columns=['price'])
    y_ = frame['price']
    X_tr, X_te, y_tr, y_te = train_test_split(X_, y_, test_size=0.2, random_state=random_state)
    model = XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.1,
                         subsample=0.8, colsample_bytree=0.8,
                         random_state=random_state, n_jobs=-1)
    model.fit(X_tr, y_tr)
    return y_te, model.predict(X_te)


# Step 1: row counter dropped
y_te1, pred1 = d1_variant(DATA_PATH, drop_counter=True, dedupe=False)
record(regression_report(y_te1, pred1, 'no row counter'))

In [ ]:
# Step 2: also dedupe, BEFORE the split
y_te2, pred2 = d1_variant(DATA_PATH, drop_counter=True, dedupe=True)
record(regression_report(y_te2, pred2, 'no counter + deduped'))

### 3c — Step 3: a `Pipeline`, so leakage becomes structurally impossible

A **`Pipeline`** is a list of `(name, transform)` steps ending in a model, wrapped to behave as one estimator. What matters is what `.fit()` does:

```
pipe.fit(X_train, y_train)
   -> every transform learns ONLY from X_train
pipe.predict(X_test)
   -> every transform APPLIES what it learned; it never re-learns from X_test
```

You cannot accidentally fit on the test set, because you never call `.fit` on a transform yourself. And this becomes essential rather than merely tidy in Section 4, where we cross-validate over many `alpha` values: without a pipeline, the scaler would be refitted on each fold's test data and every number would be slightly wrong.

A **`ColumnTransformer`** routes different columns to different transforms:

```python
ColumnTransformer([
    ('nom', OneHotEncoder(handle_unknown='ignore'), nominal_cols),
    ('ord', OrdinalEncoder(categories=[...]),       ordinal_cols),
    ('num', StandardScaler(),                       numeric_cols),
])
```

`handle_unknown='ignore'` is the production-mindset flag: if an unseen category appears at prediction time (a new airline launches), encode it as all-zeros instead of crashing.

In [ ]:
NOMINAL = ['airline', 'source_city', 'destination_city', 'class']
ORDINAL = ['stops', 'departure_time', 'arrival_time']
NUMERIC = ['duration', 'days_left']

STOPS_ORDER = ['zero', 'one', 'two_or_more']
TIME_ORDER  = ['Early_Morning', 'Morning', 'Afternoon', 'Evening', 'Night', 'Late_Night']

> **⚠️ `remainder='drop'` is a footgun, and this is the moment to respect it.** The `ColumnTransformer` you are about to build **silently discards** every column you did not put in a group. No error. No warning.
>
> On this dataset that mistake is expensive in a very specific way: `class` is the strongest single predictor here, worth roughly **0.98 R² with it versus 0.19 without**. Forget it and your model is near-useless — and nothing in the output will say so. You will just see a bad score and start blaming your hyperparameters.
>
> So do not verify coverage by eye. Make the computer do it.

In [ ]:
assigned = set(NOMINAL) | set(ORDINAL) | set(NUMERIC)
features = set(df.columns) - {'price'}      # df has the same columns as `clean`
dropped  = features - assigned

print('assigned to a group :', sorted(assigned))
print('will be DROPPED     :', sorted(dropped))

assert dropped == {'flight'}, f'unexpectedly dropping: {sorted(dropped - {"flight"})}'
print('coverage OK')

In [ ]:
prep_tree = ColumnTransformer([
    ('nom', OneHotEncoder(handle_unknown='ignore'), NOMINAL),
    ('ord', OrdinalEncoder(categories=[STOPS_ORDER, TIME_ORDER, TIME_ORDER],
                           handle_unknown='use_encoded_value',
                           unknown_value=-1), ORDINAL),
    ('num', 'passthrough', NUMERIC),
], remainder='drop')

prep_linear = ColumnTransformer([
    ('nom', OneHotEncoder(handle_unknown='ignore'), NOMINAL),
    ('ord', OrdinalEncoder(categories=[STOPS_ORDER, TIME_ORDER, TIME_ORDER],
                           handle_unknown='use_encoded_value',
                           unknown_value=-1), ORDINAL),
    ('num', StandardScaler(), NUMERIC),
], remainder='drop')

In [ ]:
# 'flight' is dropped from the model, so two rows identical in every USED column are
# duplicates to the model even when their flight codes differ. Dedupe on those columns.
feature_cols = [c for c in df.columns if c != 'flight']
clean = df.drop_duplicates(subset=feature_cols).reset_index(drop=True)
print('duplicates removed:', len(df) - len(clean))
X = clean.drop(columns=['price'])
y = clean['price']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE)
print(f'{len(X_train):,} train / {len(X_test):,} test')

xgb_pipe = Pipeline([('prep', prep_tree),
                     ('model', XGBRegressor(n_estimators=300, max_depth=6,
                                            learning_rate=0.1, subsample=0.8,
                                            colsample_bytree=0.8,
                                            random_state=RANDOM_STATE,
                                            n_jobs=-1))])
xgb_pipe.fit(X_train, y_train)
record(regression_report(y_test, xgb_pipe.predict(X_test), 'XGBoost (no leaks, pipeline)'))

### 3d — Target leakage: the expensive kind

The three leaks above were about *process*. This one is about a *feature*, and it is the one that ends projects.

Here is a feature a well-meaning teammate will propose:

```python
df['price_per_hour'] = df['price'] / df['duration']
```

It is intuitive, it is a real industry unit (yield per block hour), and adding it will take your R² to something like 0.999.

**It is computed from the target.** At prediction time you do not have `price` — predicting it is the whole job. The feature can only exist for rows where you already know the answer.

**Target leakage** is the most expensive mistake in applied ML precisely because it does not look like a bug. Nothing crashes, no warning fires, your metrics improve, and the model is worthless.

> **The test, five seconds per feature:** *"at the moment I need a prediction, do I have this value?"* If the honest answer is "no, I would need the price first" — delete it.

In [ ]:
leaky = clean.copy()
leaky['price_per_hour'] = leaky['price'] / leaky['duration']
Xl = leaky[['price_per_hour', 'duration', 'days_left']]
yl = leaky['price']
Xl_tr, Xl_te, yl_tr, yl_te = train_test_split(Xl, yl, test_size=0.2,
                                              random_state=RANDOM_STATE)
m = XGBRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
m.fit(Xl_tr, yl_tr)
print(regression_report(yl_te, m.predict(Xl_te), 'TARGET LEAK -- do not ship'))

### 3e — One split is an anecdote

Even leak-free, our score rests on **one** arbitrary 80/20 split. Change `random_state` and the number moves. How much is measurable, and you should measure it before quoting anything.

**K-fold cross-validation** splits the data into k parts, trains k times, validates on each part once. The **mean** is a better estimate; the **standard deviation** tells you how much your single number could have been luck.

> Quote scores as **mean ± std**. A model at 0.95 ± 0.002 and one at 0.95 ± 0.04 are not the same model, and the second is not ready to ship.

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
t0 = time.time()
scores = cross_validate(xgb_pipe, X, y, cv=cv,
                        scoring=['r2', 'neg_mean_absolute_error'],
                        n_jobs=1, return_train_score=True)
print(f'5-fold CV took {time.time() - t0:.0f}s')

cv_r2  = scores['test_r2']
cv_mae = -scores['test_neg_mean_absolute_error']
print(f'test R2 : {cv_r2.mean():.4f} +/- {cv_r2.std():.4f}   (min {cv_r2.min():.4f}, max {cv_r2.max():.4f})')
print(f'test MAE: {cv_mae.mean():.0f} +/- {cv_mae.std():.0f}   (min {cv_mae.min():.0f}, max {cv_mae.max():.0f})')
print(f'train R2: {scores["train_r2"].mean():.4f}   vs test R2: {cv_r2.mean():.4f}')

**The leak ledger.** *(4–6 sentences.)*

- Which single fix cost the most R²? Why that one?
- Did your honest number end up lower than your Deliverable 1 number, higher, or about the same? **Whichever it is, which number would you put in a report to American Airlines, and how would you explain the difference to someone who already saw the first one?** Note that "the leak was worth almost nothing here" is a perfectly good answer — if that is what you measured, say it and explain why.
- What does your CV standard deviation tell you about how much to trust one decimal place of these scores?

- **Which fix cost the most?** Dropping the row counter: R² fell from **0.9777 to 0.9738** (MAE ₹1,934 → ₹2,116). It carried signal because the file is sorted by class (Economy rows 0–206,665, Business rows 206,666 onwards), so the row number was a proxy for `class` and ranked third in importance (0.023). Deduplicating added about +0.0004 R² (0.9742), and the pipeline with proper encodings recovered some (0.9759, MAE ₹2,002).
- **Honest vs D1:** the honest number (**R² ≈ 0.976, MAE ≈ ₹2,000**) is slightly lower than the D1 number (0.9777, MAE ₹1,934). I would report the honest one to American Airlines and explain that the earlier figure had a row-number shortcut in it. Here the leaks were worth little (about 0.002 R²), because `class` already explains most of the price, so "the leak barely mattered here" is the real finding, not a failure.
- **Target leakage:** `price_per_hour` gave R² 0.9996 and MAE ₹239, a fake result, because it is computed from the target.
- **CV:** 5-fold R² is **0.9763 ± 0.0002** and MAE ₹1,993 ± 4, with train R² (0.9775) barely above test. So the score is stable to about the third decimal place, and the single-split number is not a fluke.

---
# Section 4 — Lasso vs Ridge: two ways to regularise a linear model

**Goal:** understand what regularisation *is*, what the L1 and L2 penalties do differently, and why that difference is useful rather than academic.

### Why fit a linear model at all, when you already have XGBoost?

Three reasons, and none of them is "it might win":

1. **It is a reference point.** "XGBoost got R² 0.95" is meaningless until you know a linear model gets 0.90 — or 0.55.
2. **It is interpretable.** A linear model hands you one coefficient per feature: *this airline adds ₹1,800 to the fare, all else equal.* No tree ensemble gives you a sentence like that, and that sentence is often what a client actually wants.
3. **Lasso does feature selection for free**, which is the real payoff in this section.

### The shared objective

Both models minimise squared error **plus** a penalty on the size of the coefficients:

```
Ridge  (L2):   minimise   ||y - Xw||²  +  α · Σ wⱼ²
Lasso  (L1):   minimise   ||y - Xw||²  +  α · Σ |wⱼ|
```

The only difference is `wⱼ²` versus `|wⱼ|`. That one change produces completely different behaviour:

| | **Ridge (L2)** | **Lasso (L1)** |
|---|---|---|
| Penalty | sum of **squared** coefficients | sum of **absolute** coefficients |
| Shrinks coefficients | yes, smoothly toward zero | yes |
| Sets them to **exactly zero** | essentially never | **yes** — that is the point |
| Correlated features | **shares** the weight between them | **picks one**, zeroes the rest |
| Gives you | a stable model using every feature | a **sparse** model, i.e. feature selection |
| Closed-form solution | yes | no — needs iterative optimisation |
| Reach for it when | you believe most features matter a little | you believe few features matter a lot |

### Why does L1 produce exact zeros and L2 does not?

The geometric picture is the one worth remembering. Think of the penalty as a budget constraining where `w` can live:

- The **L2** budget `Σwⱼ² ≤ t` is a **circle** (sphere). Smooth, no corners.
- The **L1** budget `Σ|wⱼ| ≤ t` is a **diamond**, and its corners sit **on the axes**.

The solution lands where the error contours first touch that region. A diamond's corners stick out, so contours tend to touch at a corner — and a corner *is* a point where some coefficient equals exactly zero. A circle has no corners, so the touch point almost always has every coefficient slightly non-zero.

> **⚠️ Scaling is mandatory here, not optional.** Both penalties sum over coefficient *magnitudes*, and a coefficient's magnitude depends on its feature's units. `duration` in hours (0–50) and `days_left` in days (1–49) are comparable, but if one were in minutes its coefficient would be ~60× smaller and the penalty would barely touch it. **Unscaled features mean the penalty silently punishes some features more than others.** This is why `prep_linear` has a `StandardScaler` and `prep_tree` does not — and why both live inside a `Pipeline`, so the scaler is fitted per fold.

In [ ]:
models = {
    'LinearRegression (no penalty)': LinearRegression(),
    'Ridge (alpha=1)': Ridge(alpha=1.0, random_state=RANDOM_STATE),
    'Lasso (alpha=1)': Lasso(alpha=1.0, random_state=RANDOM_STATE, max_iter=5000),
}
for name, est in models.items():
    pipe = Pipeline([('prep', prep_linear), ('model', est)])
    t0 = time.time()
    pipe.fit(X_train, y_train)
    rep = regression_report(y_test, pipe.predict(X_test), name)
    rep['fit_seconds'] = round(time.time() - t0, 1)
    record(rep)
show_scoreboard().tail(3)

### 4a — What `alpha` actually does

`alpha` is the **regularisation strength** — how much you charge the model for large coefficients.

- `alpha → 0`: no penalty. The model fits the training data as hard as it can. **High variance**, overfits.
- `alpha → ∞`: the penalty dominates. All coefficients are crushed toward zero and the model predicts roughly a constant. **High bias**, underfits.

Somewhere in between is the right value, and you find it by validation, not by taste. This is the **bias–variance tradeoff** with a dial on it.

In [ ]:
import warnings
from sklearn.exceptions import ConvergenceWarning

alphas = np.logspace(-3, 3, 13)      # 0.001 ... 1000, log-spaced
rows = []
with warnings.catch_warnings():
    warnings.simplefilter('ignore', ConvergenceWarning)   # small-alpha Lasso is slow to settle
    for a in alphas:
        for name, est in [('Ridge', Ridge(alpha=a)),
                          ('Lasso', Lasso(alpha=a, max_iter=5000))]:
            pipe = Pipeline([('prep', prep_linear), ('model', est)])
            pipe.fit(X_train, y_train)
            rows.append({'alpha': a, 'model': name,
                         'train_r2': pipe.score(X_train, y_train),
                         'test_r2': pipe.score(X_test, y_test)})
sweep = pd.DataFrame(rows)

for name, grp in sweep.groupby('model'):
    plt.semilogx(grp['alpha'], grp['test_r2'], marker='o', label=f'{name} test')
    plt.semilogx(grp['alpha'], grp['train_r2'], ls='--', alpha=0.5, label=f'{name} train')
plt.xlabel('alpha (regularisation strength)'); plt.ylabel('R2'); plt.legend()
plt.title('R2 vs alpha'); plt.show()

best = sweep.loc[sweep.groupby('model')['test_r2'].idxmax()]
print(best.round(4).to_string(index=False))

In [ ]:
lasso_pipe = Pipeline([('prep', prep_linear),
                       ('model', Lasso(alpha=100, max_iter=20000))])
lasso_pipe.fit(X_train, y_train)
ridge_pipe = Pipeline([('prep', prep_linear), ('model', Ridge(alpha=100))])
ridge_pipe.fit(X_train, y_train)

names = lasso_pipe.named_steps['prep'].get_feature_names_out()
coefs = pd.Series(lasso_pipe.named_steps['model'].coef_, index=names)
ridge_coefs = pd.Series(ridge_pipe.named_steps['model'].coef_, index=names)

print('Lasso(alpha=100) test R2:', round(lasso_pipe.score(X_test, y_test), 4))
print('Ridge(alpha=100) test R2:', round(ridge_pipe.score(X_test, y_test), 4))
print('Lasso features kept  :', (coefs != 0).sum(), '| zeroed:', (coefs == 0).sum())
print('Ridge features kept  :', (ridge_coefs != 0).sum(), '| zeroed:', (ridge_coefs == 0).sum())
print()
print('zeroed by Lasso:', list(coefs[coefs == 0].index))
print()
coefs[coefs != 0].sort_values(key=abs, ascending=False).head(15).round(0)

In [ ]:
for a in [1.0, 0.01]:
    for l1 in [0.1, 0.5, 0.9]:
        pipe = Pipeline([('prep', prep_linear),
                         ('model', ElasticNet(alpha=a, l1_ratio=l1, max_iter=5000))])
        with warnings.catch_warnings():
            warnings.simplefilter('ignore', ConvergenceWarning)
            pipe.fit(X_train, y_train)
        rep = regression_report(y_test, pipe.predict(X_test), f'ElasticNet (alpha={a}, l1_ratio={l1})')
        print(rep['model'], '-> R2', round(rep['R2'], 4), '| MAE', round(rep['MAE']))

**Lasso vs Ridge: your verdict.** *(5–7 sentences.)*

1. Which linear model scored best, and at what alpha?
2. How many features did Lasso zero out? Name two it discarded and say whether that was reasonable given your Section 2 EDA.
3. Did `LinearRegression` with no penalty do noticeably worse than the regularised versions? With ~300k rows and only ~30 one-hot columns, you have far more data than parameters — explain why that makes regularisation matter *less* here than it would on a wide, small dataset. **This is the honest answer, and it is more useful than pretending regularisation saved you.**
4. If a PM asked for "a model I can explain to a non-technical stakeholder", which of your models so far would you hand over, and what would you lose by doing so?

1. **Best linear model:** all three are tied. LinearRegression, Ridge (α=1) and Lasso (α=1) all reach **R² 0.9092, MAE ≈ ₹4,517**. In the sweep the best Lasso was at α=1 and the best Ridge at α=100, but the differences were in the fourth decimal.
2. **Lasso zeroed 9 of 25 features at α=100** (R² 0.9074). Among them: `airline_Indigo`, `airline_SpiceJet`, `airline_GO_FIRST`, and the Bangalore, Chennai and Mumbai city columns. That is reasonable: the three budget airlines only sell Economy (my 2d crosstab), so class already explains most of their price, and the route averages in 2d only varied between about ₹17,000 and ₹25,000. Ridge at the same α zeroed **0**, which shows the L1/L2 difference directly.
3. **No penalty did not do noticeably worse.** With about 238,000 rows and only about 25 columns there is far more data than parameters, so overfitting is not the problem and regularisation barely matters. ElasticNet at α=1 did badly (R² 0.52–0.88) only because its L2 part crushed the huge `class` coefficient (the target is in rupees); at α=0.01 it was back to 0.909. Linear models are limited by *bias* here (R² 0.909 vs 0.976 for XGBoost; MAPE 43% vs 14.5%), not variance.
4. **For a non-technical stakeholder** I would hand over the linear model (Ridge): it gives sentences like the ones I read off the Lasso coefficients, "Business adds about ₹44,650, each step up in stops adds about ₹4,900, Vistara adds about ₹2,600, all else equal". What I would lose is accuracy: its MAE is about 2.3× worse than XGBoost and it has a 43% average percentage error.

---
# Section 5 — Loss functions: what are you actually asking the model to minimise?

**Goal:** understand that "train a model" means "minimise a specific definition of wrong", that you *choose* that definition, and that the default is not always right.

Every model in Sections 3 and 4 minimised **squared error**, because that is the default and nobody told you otherwise. That was a choice, made silently, and it had consequences you can measure.

### Loss vs metric — keep these straight

| | **Loss function** | **Evaluation metric** |
|---|---|---|
| Used | *during* training, by the optimiser | *after* training, by you |
| Must be | differentiable (for gradient methods) | anything computable |
| Purpose | tells the model which direction to move | tells you how good the result is |
| Example | `reg:squarederror` | R², MAPE |

They do not have to match — but if your loss and your metric disagree about what "wrong" means, you are optimising for one thing and grading on another. That is worth doing deliberately, never by accident.

### The four losses that matter for regression

Let the error be `e = ŷ - y`.

| Loss | Formula | Penalises | Predicts the | Outlier behaviour |
|---|---|---|---|---|
| **Squared / L2 (MSE)** | `e²` | large errors **quadratically** | conditional **mean** | very sensitive — one bad row can dominate |
| **Absolute / L1 (MAE)** | `|e|` | errors **linearly** | conditional **median** | robust |
| **Huber** | `e²` for small `|e|`, linear beyond `δ` | both, by region | something between | robust, and smooth at zero |
| **Quantile / pinball** | `α·e` if `e<0`, `(α−1)·e` if `e>0` | over- and under-prediction **differently** | the **α-th quantile** | tunable, asymmetric |

Two consequences worth internalising:

- **MSE predicts the mean; MAE predicts the median.** That is not a detail. On the right-skewed price distribution you plotted in Section 2a, the mean sits well above the median, so these two losses produce *systematically different* predictions on the same data.
- **Huber exists because MSE and MAE each have one flaw.** MSE overreacts to outliers; MAE has a kink at zero that makes optimisation less stable and treats a ₹1 error with the same gradient as a ₹10,000 one. Huber is quadratic near zero and linear in the tails — and `δ` is where it switches.

> ### ⚠️ ReLU is an activation function, not a loss function
>
> This confusion is extremely common and worth clearing up precisely, because the two things live in different places in a model.
>
> **ReLU** is `f(x) = max(0, x)`. It is an **activation function**: it sits *inside* a neural network, transforming one layer's output before it reaches the next. It takes one number and returns one number. It has no idea what the true label is, so **it cannot be a loss** — a loss must compare a prediction to a truth.
>
> ```
> input -> [linear layer] -> ReLU -> [linear layer] -> ReLU -> output -> LOSS(output, truth)
>                             ^^^^                     ^^^^              ^^^^
>                        activations: shape the model   |          loss: scores the model
> ```
>
> **But `max(0, ·)` genuinely does appear inside loss functions**, which is probably where the confusion comes from — and this part is useful:
>
> - **Hinge loss** (classification, SVMs): `max(0, 1 − y·ŷ)`. Zero penalty once you are correct by a margin, linear penalty otherwise. That is literally a ReLU applied to the margin.
> - **Pinball / quantile loss** (regression, below): `max(α·e, (α−1)·e)` — two ReLU-shaped arms joined at zero, with **different slopes** on each side. The asymmetry of those slopes is the whole mechanism.
>
> So: ReLU is a **building block** of asymmetric and margin-based losses, not a loss itself. If a teammate says "let's use ReLU as our loss", the useful reply is "do you mean a hinge or a quantile loss?" — because that is almost always the idea underneath.

### 5a — Quantile loss, and why it is the interesting one for pricing

Every loss above is **symmetric**: over-predicting by ₹500 costs exactly what under-predicting by ₹500 costs.

**Pricing is not symmetric.** Consider the two ways to be wrong:

- **Quote too low.** You sell the seat, but you left money on the table — a loss of margin on one booking.
- **Quote too high.** The customer books a competitor. You lose the entire sale, and possibly the customer.

Those costs are not equal, and which is worse depends on the business situation — a nearly-full flight versus one departing half-empty. A symmetric loss cannot express any of that. **Quantile loss can**, because `α` sets the asymmetry:

- `α = 0.5` → symmetric → predicts the **median** (this is exactly MAE)
- `α = 0.9` → under-prediction punished 9× harder → predicts the **90th percentile**, a deliberately high quote
- `α = 0.1` → over-prediction punished 9× harder → predicts the **10th percentile**, a deliberately low quote

Fit three of them and you get a **prediction interval** rather than a single number — "this fare is most likely ₹6,400, and we are fairly confident it is between ₹4,900 and ₹8,800". For a pricing recommendation, that interval is far more useful than a point estimate, because it tells the user how much to trust it.

In [ ]:
print('xgboost', xgboost.__version__)
models = {
    'MSE  (reg:squarederror)':  XGBRegressor(objective='reg:squarederror',
                                    n_estimators=300, max_depth=6, learning_rate=0.1,
                                    random_state=RANDOM_STATE, n_jobs=-1),
    'MAE  (reg:absoluteerror)': XGBRegressor(objective='reg:absoluteerror',
                                    n_estimators=300, max_depth=6, learning_rate=0.1,
                                    random_state=RANDOM_STATE, n_jobs=-1),
    'Huber (sklearn GBR)':      GradientBoostingRegressor(loss='huber', alpha=0.9,
                                    n_estimators=200, max_depth=5,
                                    random_state=RANDOM_STATE),
}
for name, est in models.items():
    pipe = Pipeline([('prep', prep_tree), ('model', est)])
    t0 = time.time()
    pipe.fit(X_train, y_train)
    rep = regression_report(y_test, pipe.predict(X_test), name)
    rep['fit_seconds'] = round(time.time() - t0, 1)
    record(rep)
show_scoreboard().tail(3)

In [ ]:
# The huber_slope lesson: same objective, two slopes. Not recorded -- a demonstration.
for slope, start in [(1, None), (100, None), (5000, None), (5000, float(y_train.mean()))]:
    p = Pipeline([('prep', prep_tree),
                  ('model', XGBRegressor(objective='reg:pseudohubererror', huber_slope=slope,
                                         base_score=start,
                                         n_estimators=300, max_depth=6, learning_rate=0.1,
                                         random_state=RANDOM_STATE, n_jobs=-1))])
    p.fit(X_train, y_train)
    label = f'pseudohuber slope={slope}' + (', start=mean price' if start else '')
    r = regression_report(y_test, p.predict(X_test), label)
    print(r['model'], '-> R2', round(r['R2'], 4), '| MAE', round(r['MAE']))

# Three quantile models -> a prediction interval
quantiles = [0.1, 0.5, 0.9]
q_preds = {}
for q in quantiles:
    pipe = Pipeline([('prep', prep_tree),
                     ('model', GradientBoostingRegressor(
                         loss='quantile', alpha=q, n_estimators=150,
                         max_depth=5, random_state=RANDOM_STATE))])
    pipe.fit(X_train, y_train)
    q_preds[q] = pipe.predict(X_test)
    print(f'q={q} fitted')

inside = ((y_test.to_numpy() >= q_preds[0.1]) & (y_test.to_numpy() <= q_preds[0.9])).mean()
print(f'coverage of the 10-90 interval: {100 * inside:.1f}%  (target ~80%)')
for q in quantiles:
    print(f'pinball loss q={q}: {mean_pinball_loss(y_test, q_preds[q], alpha=q):.1f}')

# interval for ~40 test rows, sorted by predicted median
idx = np.argsort(q_preds[0.5])[::len(q_preds[0.5]) // 40][:40]
xs = np.arange(len(idx))
plt.fill_between(xs, q_preds[0.1][idx], q_preds[0.9][idx], alpha=0.3, label='10-90% band')
plt.plot(xs, q_preds[0.5][idx], label='median prediction')
plt.scatter(xs, y_test.to_numpy()[idx], color='k', s=15, label='actual')
plt.xlabel('test rows sorted by predicted median'); plt.ylabel('price (INR)')
plt.legend(); plt.title('Prediction interval from quantile models'); plt.show()

**Loss functions: your verdict.** *(5–7 sentences.)*

1. Which objective won on which metric? State the pattern you observed in one sentence.
2. Given the price distribution from Section 2a — two populations, strongly right-skewed overall — which loss do you think is the right default for *this* dataset, and why?
3. What was your 80% interval's actual coverage? If it missed the target, is the model over- or under-confident?
4. **The business question.** For a fare-recommendation tool, is it worse to suggest a price that is too high or too low? Pick a side, commit to it, and say which `α` you would ship. There is no neutral answer — the symmetric default *is* a choice, it is just an unexamined one.

1. **Pattern:** the squared-error model won on RMSE (3,504) and R² (0.9761), the absolute-error model won on **MAPE% (12.9 vs 14.5)**, and Huber (sklearn GBR) was last on MAE (₹2,350) and slowest (about 80s).
2. **Surprise:** the MAE-trained model did *not* win on MAE (₹2,223 vs ₹2,000 for MSE), which the textbook expects. A likely reason is that `reg:absoluteerror` takes constant-size gradient steps and is slower to converge, so at 300 trees it is probably under-trained, and this price distribution is mostly two clean populations rather than a few wild outliers, which is where MAE loss helps most. This is a finding about my data and settings, not proof that MAE loss is worse.
3. **Right default:** for rupee error, plain squared error was fine here. For the business goal (relative error), the MAE-type loss gave the lowest MAPE because it does not over-prioritise the large Business fares. I would keep squared error for a point model and report MAE and MAPE, and use quantile loss for the pricing tool.
4. **Interval coverage:** the 10–90 interval covered **79.8%** of test prices against a target of 80%, so the model is well calibrated on this split. Pinball loss was 519 (α=0.1), 1,265 (0.5) and 675 (0.9).
5. **The huber_slope lesson:** with `reg:pseudohubererror` and the default slope of 1, training diverged (R² ≈ −1.3e13). Larger slopes helped but never converged: slope 100 gave R² −130,000, slope 5,000 gave −221, and starting from the mean price gave −38. The slope is in rupees, so it must match the target's scale, and it also interacts with the starting point and learning rate. I did not fully solve this one; scaling the target (e.g. log price) would be my next step.
6. **Business question:** I would say **quoting too high is worse than too low**: a quote that is too high loses the whole sale and possibly the customer, while one that is too low only loses some margin. The optimal quantile is α = Cu/(Cu+Co), where Cu is the cost of under-quoting and Co the cost of over-quoting. If I assume over-quoting costs about 3× as much, α ≈ 0.25, so I would ship a quantile near **0.25–0.3** rather than the median. The 3× figure is my assumption, not data.

---
# Section 6 — XGBoost vs Random Forest vs ARIMA

**Goal:** choose a model family on the basis of what the problem is, not what the leaderboard says.

These three are not three contestants in one race. **Two of them answer the same question in different ways, and the third answers a different question entirely** — recognising that is the actual skill in this section.

| Model | Family | What it needs | What it predicts |
|---|---|---|---|
| **Random Forest** | bagged trees | rows of features, any order | one target value **per row** |
| **XGBoost** | boosted trees | rows of features, any order | one target value **per row** |
| **ARIMA** | time series | one **ordered sequence**, equally spaced | the **next values** of that sequence |

Random Forest and XGBoost are interchangeable in their interface: both take `X, y` and predict a price for an individual ticket. ARIMA cannot do that at all — it does not take features. It takes a *sequence* and extends it.

---

### 6a — Bagging vs boosting: how RF and XGBoost actually differ

Both build many decision trees. **How** they build them is close to opposite.

| | **Random Forest (bagging)** | **XGBoost (boosting)** |
|---|---|---|
| Trees are built | **independently**, all in parallel | **sequentially**, each one after the last |
| Each tree is trained on | a bootstrap sample + a random feature subset | the **errors (gradients)** the ensemble still has |
| Individual trees are | deep, overfit, **decorrelated** | shallow, weak learners |
| Combined by | **averaging** | **adding** (scaled by the learning rate) |
| Mainly reduces | **variance** | **bias** |
| More trees → | never really worse, just slower | **can overfit**; needs `learning_rate` / early stopping |
| Tuning burden | low — works well out of the box | higher — more knobs that interact |
| Parallelises | trivially (trees are independent) | within a tree, not across them |

The intuition: **bagging** builds many different experts and averages away their individual mistakes. **Boosting** builds one expert, finds what it gets wrong, and hires a specialist for exactly that — repeatedly.

This explains their failure modes. A Random Forest rarely blows up but often plateaus slightly below a well-tuned booster. XGBoost usually wins *after tuning* and overfits cheerfully if you leave `n_estimators` high and `learning_rate` high.

In [ ]:
contenders = {
    'Dummy (mean)': DummyRegressor(strategy='mean'),
    'RandomForest': RandomForestRegressor(
        n_estimators=100, max_depth=None, min_samples_leaf=2,
        random_state=RANDOM_STATE, n_jobs=-1),
    'XGBoost': XGBRegressor(
        n_estimators=300, max_depth=6, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        random_state=RANDOM_STATE, n_jobs=-1),
}
fitted = {}
for name, est in contenders.items():
    pipe = Pipeline([('prep', prep_tree), ('model', est)])
    t0 = time.time()
    pipe.fit(X_train, y_train)
    fitted[name] = pipe
    rep = regression_report(y_test, pipe.predict(X_test), name)
    rep['fit_seconds'] = round(time.time() - t0, 1)
    record(rep)
print(show_scoreboard().tail(3))

# Diagnostic (not recorded): is XGBoost UNDER-fitting? Compare train vs test R2, then
# give it more capacity. Tuning properly is Deliverable 3 -- this only explains the gap.
deep = Pipeline([('prep', prep_tree),
                 ('model', XGBRegressor(n_estimators=600, max_depth=12, learning_rate=0.1,
                                        subsample=0.8, colsample_bytree=0.8,
                                        random_state=RANDOM_STATE, n_jobs=-1))])
deep.fit(X_train, y_train)
fitted['XGBoost (depth 12, 600 trees)'] = deep
print()
for name in ['RandomForest', 'XGBoost', 'XGBoost (depth 12, 600 trees)']:
    pipe = fitted[name]
    print(f'{name:32s} train R2 {pipe.score(X_train, y_train):.4f} | test R2 {pipe.score(X_test, y_test):.4f}'
          f' | test MAE {mean_absolute_error(y_test, pipe.predict(X_test)):.0f}')

**RF vs XGBoost.** *(4–6 sentences.)*

- Which won, and by how much in **MAE rupees** (not R² — rupees is the version a stakeholder understands)?
- How did fit times compare? If one took 10× longer for 1% more R², which would you ship, and what does that depend on?
- How far above the `DummyRegressor` is each? That gap is how much your features are actually worth.
- Which would you rather hand to a teammate who has never tuned a model, and why?

- **Winner:** Random Forest, with MAE **₹1,088** vs **₹2,002** for XGBoost (46% lower), R² 0.9862 vs 0.9759, and MAPE 7.3% vs 14.5%.
- **Fit time:** RF took 12.5s against 1.7s for XGBoost (about 7× longer).
- **Why not just "RF is better":** XGBoost's train R² (0.9772) is almost the same as its test R² (0.9759), which is a sign of *under*-fitting. The unbounded forest can model high-order interactions (airline × route × class × days_left). When I gave XGBoost more capacity (depth 12, 600 trees), it reached R² **0.9882** and MAE ₹1,115, slightly above RF in R² and about equal in MAE. So the gap came from my untuned XGBoost settings, not from the model family.
- **Dummy:** the mean predictor has MAE ₹19,780 and R² 0, so both ensembles explain nearly all the useful variation (the features are worth about ₹18,000 of MAE).
- **For a teammate who has never tuned a model:** Random Forest, because it worked well out of the box with few knobs, while XGBoost needed a deeper setting to catch up.

### 6b — ARIMA: the right tool for a question we barely have

**ARIMA** = **A**uto**R**egressive **I**ntegrated **M**oving **A**verage, written `ARIMA(p, d, q)`:

- **AR(p)** — regress the value on its own previous `p` values. "Today looks like yesterday."
- **I(d)** — **difference** the series `d` times to make it **stationary** (constant mean/variance over time). A series that trends upward forever cannot be modelled directly; its *changes* usually can.
- **MA(q)** — regress on the previous `q` forecast *errors*. Lets recent shocks decay.

ARIMA is the standard tool for forecasting a single quantity through time: monthly revenue, daily passengers, weekly bookings.

### So does it apply to this dataset?

Be honest about this, because it is the real lesson:

**Mostly, no — and that is a legitimate finding.** ARIMA needs a series indexed by time, and this dataset **has no dates**. There is no departure date, no booking date, no timestamp anywhere. Without a calendar axis there is no seasonality, no trend over real time, no day-of-week effect. The dataset is a cross-section: 300,000 independent snapshots with no temporal order among them.

**But there is one real sequence in here**, and you plotted it in Section 2c: **mean price by `days_left`**. Days-to-departure is an ordered, equally-spaced index, so the booking curve is a genuine series of ~49 points — and forecasting the booking curve is a real revenue-management task. So we apply ARIMA to the thing it can actually model, and then we are precise about what we did and did not learn.

> **The transferable lesson:** your first model-selection question is never "which model scores best?" It is **"what shape is my data, and what shape does this model require?"** A model that needs a time index will not acquire one because you want it to. Picking the wrong family is a mistake no amount of hyperparameter tuning can repair.

In [ ]:
from statsmodels.tsa.stattools import adfuller

# Sort DESCENDING by days_left: position 0 is furthest from departure.
curve = clean.groupby('days_left')['price'].mean().sort_index(ascending=False)
series = curve.reset_index(drop=True)
print(f'{len(series)} points')

stat, p, *_ = adfuller(series)
print(f'ADF p-value (raw):         {p:.4f}')
stat_d, p_d, *_ = adfuller(series.diff().dropna())
print(f'ADF p-value (differenced): {p_d:.4f}')
stat_dd, p_dd, *_ = adfuller(series.diff().diff().dropna())
print(f'ADF p-value (differenced twice): {p_dd:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(series); axes[0].set_title('booking curve (raw)')
axes[0].set_xlabel('step toward departure'); axes[0].set_ylabel('mean price')
axes[1].plot(series.diff()); axes[1].set_title('first difference')
axes[1].set_xlabel('step toward departure')
plt.tight_layout(); plt.show()

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

train_s, test_s = series[:-9], series[-9:]

arima = ARIMA(train_s, order=(2, 1, 2)).fit()
print(arima.summary().tables[1])
fc = arima.forecast(steps=9)

naive = np.repeat(train_s.iloc[-1], 9)
print('ARIMA(2,1,2) MAE:', round(mean_absolute_error(test_s, fc)))
print('Naive MAE       :', round(mean_absolute_error(test_s, naive)))

plt.plot(train_s.index, train_s, label='train')
plt.plot(test_s.index, test_s, label='actual (held out)', color='k')
plt.plot(test_s.index, fc, label='ARIMA(2,1,2)', color='tab:red')
plt.plot(test_s.index, naive, label='naive (last value)', ls='--', color='tab:green')
plt.legend(); plt.xlabel('step toward departure'); plt.ylabel('mean price (INR)')
plt.title('Booking-curve forecast'); plt.show()

print()
print('other orders:')
for order in [(1, 1, 0), (0, 1, 1), (3, 1, 1), (2, 1, 2)]:
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        res = ARIMA(train_s, order=order).fit()
        f = res.forecast(steps=9)
    print(order, '-> AIC', round(res.aic, 1), '| MAE', round(mean_absolute_error(test_s, f)))

**ARIMA: the verdict.** *(5–7 sentences. This is the most important written answer in the notebook.)*

1. Did ARIMA beat the naive last-value forecast? By how much? If it did not, say so plainly — a negative result reported honestly is worth more than a flattering one.
2. **What question did this ARIMA model answer?** Be precise. It did *not* price a flight. Write down what it actually produces and who could use it.
3. **What did you give up to use it?** You aggregated 300,000 rows into ~49 points. Name what was thrown away. Can this model tell you anything about airline, class, route or stops?
4. Would you include ARIMA in the deliverable to American Airlines? If so, in what role — alongside XGBoost rather than instead of it? If not, what would have to change about the dataset for it to earn a place?

1. **Beat the naive forecast?** No. ARIMA(2,1,2) had MAE ₹1,654 against ₹1,647 for carrying the last value forward. Simpler orders did marginally better (e.g. (1,1,0): ₹1,626, about 1.3% better than naive), but I picked those after seeing the test error, so that gap is optimistic. The ADF test did not support stationarity at any differencing level (p = 0.91 raw, 0.27 after one difference, 0.68 after two), although with only 49 points the test has little power. I used d=1 anyway, and the MA coefficient had an enormous standard error (about 8.6), a sign the model is over-parameterised for 40 training points.
2. **What question it answered:** "what will the *average* fare be at each step toward departure?" It produces one aggregate forecast curve and could be used by a revenue-management analyst, but it did **not** price a flight.
3. **What I gave up:** I aggregated about 298,000 rows into 49 points and threw away airline, class, route, stops, and times. It cannot say anything about any of those.
4. **Would I include it?** Not as part of this deliverable to American Airlines, except possibly alongside XGBoost or RF as a sanity curve of average fares. To earn a place, the data would need real dates (departure and booking timestamps), so there is an actual time series with seasonality.

> **For the curious: SARIMAX.** The `statsmodels` family does extend to seasonality and features: `SARIMAX` adds a seasonal component and an `exog=` argument for external regressors. With real departure dates you could model weekly and holiday seasonality and feed in fuel price or capacity — which is much closer to how an airline forecasts demand. We cannot, because this dataset has no dates. Worth knowing the door exists.

### 6c — The model-selection decision table

Fill this in from your own numbers. This table, not any single score, is the deliverable for Section 6.

| Model | Best MAE (INR) | R² | Fit time | Interpretable? | When would you choose it? |
|---|---|---|---|---|---|
| Dummy (mean) | 19,780 | 0.000 | 0.3 s | — | never; it is the floor |
| Ridge | 4,518 | 0.909 | 0.3 s | yes — coefficients | a fast, explainable baseline, stable with correlated features |
| Lasso | 4,516 | 0.909 | 0.6 s | yes — sparse coefficients | when you want to know which features can be dropped (it zeroed 9 of 25 at α=100) |
| Random Forest | 1,088 | 0.986 | 12.5 s | no | a strong default with few knobs when you want accuracy without tuning |
| XGBoost | 2,002 (depth 6) / 1,115 (depth 12, 600 trees) | 0.976 / 0.988 | 1.7 s (depth 6) | no | when you will tune it: it has the highest ceiling |
| ARIMA | n/a — different target | n/a | under 1 s (not timed) | partly | forecasting an aggregate curve through real time, which needs real dates |

**The one you would ship, and the three reasons why:**

**Random Forest.**
1. **Accuracy per rupee:** MAE ₹1,088 and MAPE 7.3%, about half the error of my default XGBoost and 4× lower than Ridge.
2. **Tuning burden:** it reached this with two parameters set; XGBoost needed a much deeper setting just to catch up.
3. **Robustness and cost:** bagging is hard to overfit, and a 12.5-second fit is trivial.

**Strongest argument against:** it is not interpretable, a large forest is heavy to serve, and a deeper XGBoost matched or beat it (R² 0.9882, MAE ₹1,115), so the real winner after Deliverable 3's tuning may well be XGBoost. I would revisit the choice after tuning.

---
# Section 7 — Defend your choice

### 7a — The full scoreboard

In [ ]:
show_scoreboard().sort_values('MAE')

print()
print('Best model: RandomForest, MAE ~ INR 1,088 (single split; XGBoost 5-fold CV is stable to +/- 4 INR), '
      'vs INR 19,780 for the dummy.')

### 7b — What this means for American Airlines

We are modelling **Indian domestic fares scraped from EaseMyTrip in 2022**. American Airlines is a different carrier in a different market in a different year. Be precise about what transfers.

**What transfers — the structure of the problem and the method.** Airline revenue management has the same shape everywhere: fares move with the booking horizon, cabin class dominates price, route density and competition set the base level, and time-of-day and connections differentiate the product. Your model-selection reasoning transfers completely: tree ensembles for per-ticket prediction, asymmetric loss where over- and under-pricing cost differently, a time-series model for forecasting an aggregate curve through real time.

**What does not transfer:** nearly all the specifics. Different regulatory regime, competitive set, cost base, currency, and demand seasonality — and 2022 was still distorted by post-pandemic recovery.

**What the dataset is missing that real pricing requires:**

| Missing | Why it matters |
|---|---|
| **Seat inventory / load factor** | Revenue management is fundamentally about *remaining seats*. We have no capacity data at all — the largest single gap. |
| **Departure dates** | We have `days_left` but not *when*. No seasonality, no holidays, no day-of-week. **This is also precisely why ARIMA had so little to work with.** |
| **Competitor fares** | Fares are set against what everyone else charges on the route, in near-real-time. |
| **Fare class / fare basis** | "Economy" spans a dozen booking classes with different prices and restrictions; our `class` flattens all of it to two values. |
| **Demand signals** | Searches, booking pace, events — the leading indicators pricing teams actually watch. |

### 7c — Reflection

Answer each in the cell below it, in your own words. A PM will ask these in your PR review, so treat it as practice for that conversation.

---

**Q1 — The honest number.** You produced several R² values for the same task. List them with one line each on what changed, then name the single number you would put in front of American Airlines and defend it in two sentences.

**Answer Q1:** - **0.9777**: Deliverable 1, with all three leaks (row counter, duplicates across the split, encoders fitted before the split).
- **0.9738**: row counter dropped.
- **0.9742**: plus duplicates removed before the split.
- **0.9759**: the leak-free pipeline (ordinal and one-hot encodings fitted inside the split).
- **0.9763 ± 0.0002**: the same pipeline in 5-fold CV.
- (0.9996 for `price_per_hour`, which is target leakage and not a real result.)

The number I would put in front of American Airlines is the **cross-validated 0.976 ± 0.0002 (MAE about ₹2,000)** for the leak-free pipeline, because it is measured only on data the model did not see and the small ± shows it is stable. I would explain that the earlier 0.978 included a row-number shortcut. The Random Forest's 0.986 is from a single split and would need cross-validation before I quote it.

**Q2 — Model family.** You would ship one of Ridge / Lasso / Random Forest / XGBoost. Name it, and give **three** reasons that are not "it had the best score" — e.g. fit time, tuning burden, interpretability, robustness, what the stakeholder needs. Then name the strongest argument *against* your choice.

**Answer Q2:** I would ship the **Random Forest**. Three reasons that are not just the score: (1) it needed almost no tuning, with two settings, to reach MAE ₹1,088, while XGBoost needed a deeper configuration; (2) it is robust, because averaging independent trees is hard to overfit, so there is less that can go wrong in production; (3) a 12.5-second fit is cheap enough to retrain often. The strongest argument against it is that a deeper XGBoost (depth 12, 600 trees) matched it (R² 0.9882, MAE ₹1,115) and a forest is bigger and slower to serve and is not interpretable, so after tuning in Deliverable 3 XGBoost may be the better pick.

**Q3 — Lasso vs Ridge.** Explain to a teammate who has not done this deliverable why L1 produces exact zeros and L2 does not. Then say which you would use on this dataset and why — including the possibility that neither penalty mattered much here, and what about this dataset makes that the case.

**Answer Q3:** Both penalties add a cost for large coefficients to the squared error. The L2 penalty (Ridge) is Σw², a smooth round budget, so the best solution almost never lands exactly on an axis and every coefficient stays slightly non-zero. The L1 penalty (Lasso) is Σ|w|, a diamond whose corners sit on the axes, so the solution often lands on a corner, where some coefficient is exactly zero. On this dataset **neither penalty mattered much**: LinearRegression, Ridge and Lasso all scored R² 0.9092, because there are about 238,000 rows and only about 25 columns, so there is plenty of data per parameter and overfitting was never the problem. I would use Ridge here (stable, readable coefficients, no accuracy lost), and only use Lasso to find features to drop: at α=100 it zeroed 9 of 25 features at a cost of about 0.002 R².

**Q4 — Loss functions.** Your Deliverable 1 model minimised squared error because that is the default. Having seen the comparison: was that the right default for this data? Which loss would you train on now, and which metric would you report? If those two differ, justify the mismatch.

**Answer Q4:** Squared error was not a bad default for rupee error: the MSE model had the best RMSE (3,504) and, surprisingly, a better MAE than the MAE-trained model (₹2,000 vs ₹2,223, probably because the absolute-error objective converges more slowly). But on **MAPE%**, the metric closest to what the business feels, the MAE-loss model won (12.9 vs 14.5), because squared error pays most attention to the large Business fares. For the point model I would still train on squared error and report **MAE in rupees and MAPE**. For a fare-recommendation tool I would train quantile models and ship an α around 0.25–0.3. The mismatch between training loss (MSE) and reported metrics (MAE, MAPE) is deliberate: the loss is chosen for easy, stable optimisation, and the metrics are what a stakeholder understands.

**Q5 — ARIMA and problem shape.** Someone proposes ARIMA for predicting individual ticket prices. Explain in three sentences why that does not work, and what they would need to change about the data or the question for a time-series model to be the right call.

**Answer Q5:** ARIMA needs one ordered, equally spaced series and predicts the *next values of that series*, but ticket prices are individual rows with features like airline, class and route, and this dataset has no dates, so there is no series of tickets to extend. ARIMA also has no way to take those features as input, so it cannot say what *this* ticket costs. For a time-series model to be the right call, the data would need real timestamps (booking and departure dates) and the question would have to change to forecasting an aggregate quantity through time, such as average fare or bookings per day.

**Q6 — Leakage.** Explain leakage to a teammate using an example from your own notebook run (not one from the text). Then name the one habit from this notebook you will carry into Deliverable 3 to prevent it.

**Answer Q6:** In my own run, `Unnamed: 0` was the row counter from the CSV. In Deliverable 1 it stayed in `X` and ranked **third** in feature importance (0.023), ahead of `duration` and `days_left`. The reason is that the file is sorted by class (Economy rows 0–206,665, Business rows from 206,666), so the row number secretly told the model whether a ticket was Business. A real flight to be priced next week has no row number, so that signal does not exist at prediction time. Dropping it cost 0.004 R², which is the size of the fake gain. The habit I will carry into Deliverable 3: before using any feature, ask "do I have this value at the moment I need a prediction?", and put every transform inside a `Pipeline` so it is fitted on the training data only.

**Q7 — Team git.** Deliverable 1 was `deliverable1/<your-name>` → `main`. Deliverable 2 is `team-##/<your-name>` → `team-##/main` → `main`. What does the extra layer buy a project with 70+ contributors? Then describe something that actually went wrong or felt awkward in your team's git this week — a real friction point is worth more here than a clean answer.

**Answer Q7:** The extra layer means the PMs review one trunk per team instead of 70+ individual PRs into `main`, and teammates review each other at a small scale before anything reaches `main`, so `main` only changes through a reviewed trunk.

What went wrong in my team: our trunk was originally the plain branch `team-04`, and git cannot also have `team-04/<name>`, because a branch cannot be a prefix of another branch. We could not create member branches until we renamed the trunk to `team-04/main`, which meant pushing the new name and deleting the old one on GitHub. That felt risky because a teammate had already merged work into `team-04`, so we checked first that no commits would be lost and kept a temporary backup branch. We hit a second problem when we merged `main` into the trunk: `main` had deleted the shared Deliverable 1 notebook that the trunk still had modified, which gave a conflict, so I brought in only the `deliverable2/` folder instead of the whole merge.

---
## Before you submit — checklist

**The notebook**

1. **Kernel → Restart & Run All.** It must run top-to-bottom with no errors. A notebook that only works in the order you happened to click cells is not reproducible.
2. Every `# TODO` cell is filled in.
3. Every markdown answer prompt has a real answer. "*(Write here)*" left anywhere is an incomplete deliverable.
4. The three summary tables are filled in: the EDA decisions (2e), the model-selection table (6c), and the scoreboard (7a).
5. **Kernel → Restart & Clear Output** before committing. Keeps the diff reviewable and prevents most notebook merge conflicts.

**Git hygiene**

6. `git status` shows **no** `.csv`. If it does, stop — do not commit, and check [`.gitignore`](../.gitignore).
7. You are on your own member branch — not your team's trunk, and definitely not `main`. Check with `git branch --show-current`.

```sh
git add deliverable2/flight_price_modeling.ipynb
git commit -m "deliverable2: EDA, leakage fixes, model selection"
git push
```

8. Open a PR from your member branch into **your team's trunk branch**. GitHub defaults the base to `main` — change it by hand.
9. Request review from the teammate assigned to you in [`README.md`](README.md), and **review the PR assigned to you**. Reviewing is half the job.
10. Once every member's PR has merged into the team trunk, the team fills in [`TEAM_REPORT.md`](TEAM_REPORT.md) together and a PM reviews the trunk → `main`.

---

You arrived with one model and one score. You are leaving with five models, four loss functions, an honest number, and — most importantly — a reason for every choice.

In Deliverable 3 we tune the winner: hyperparameter search over the pipeline you built, learning curves to distinguish a data problem from a model problem, and error analysis by segment to find out *where* it fails rather than just how much. 🛫